In [ ]:
!pip install transformers huggingface_hub torch

In [ ]:
!pip install langchain langchain-openai langchain_community chromadb langgraph faiss-gpu

In [ ]:
!pip install --upgrade "torchao>=0.16.0"

O código deste notebook foi movido, sem alterações, para módulos `.py` na pasta `src/`:

- `model_loading.py`: login no Hugging Face e carga do `tokenizer` e do `model` ajustado.
- `graph_state.py`: o `TypedDict` `GraphState`.
- `patient_database.py`: `mock_patient_db`.
- `protocols_database.py`: `mock_protocols`.
- `vectorstore.py`: embeddings, índice FAISS e `retriever`.
- `input_nodes.py`: `parse_and_validate_input` e `request_patient_id`.
- `ehr_nodes.py`: `fetch_ehr_context`.
- `retrieval_nodes.py`: `retrieve_protocols`.
- `generation_nodes.py`: `generate_llm_response`.
- `guardrail_nodes.py`: `guardrail_evaluator` e `human_validation_gate`.
- `audit_nodes.py`: `audit_logger`.
- `graph.py`: montagem e compilação do `StateGraph` (`app`).
- `examples.py`: as três execuções de exemplo.

Cada módulo importa os anteriores, então executar as células abaixo na ordem reproduz o
fluxo original. O kernel deve rodar com a raiz do repositório como diretório de trabalho.


**Pipeline Inputs & Outputs**

The clinical assistant pipeline operates as a contextualized, retrieval-augmented decision system integrating structured patient data, internal hospital protocols, and automated safety guardrails.

* **Inputs:**
* **User Query:** Clinical question or instruction from a medical professional entered via chat or voice prompt.
* **Session Metadata (Optional):** Active `patient_id` passed directly from the Electronic Health Record (EHR) interface.
* **Internal Knowledge Base:** Hospital protocols, clinical guidelines, and standard operating procedures.


* **Outputs:**
* **Contextualized Recommendation:** Medical guidance framed around patient history and institutional protocols.
* **Source Citations (Explainability):** Direct protocol and reference links used to construct the answer.
* **System Alerts:** Warnings regarding pending diagnostic tests or critical data.
* **Structured Audit Payload:** Immutable log entry containing queries, retrieved context, generated text, safety flags, and user approvals.



---

**LangGraph Workflow Architecture**

The system uses LangGraph to coordinate entity extraction, EHR database retrieval, protocol search, LLM generation, guardrail evaluation, and physician validation gates.

**Graph State Schema**

* `user_query`: Raw prompt provided by the physician.
* `patient_id`: String identifier parsed from the query or incoming session metadata.
* `missing_patient_id`: Boolean flag set if no patient context can be established.
* `patient_context`: EHR record containing medical history and pending tests.
* `retrieved_docs`: Chunks and citations retrieved from internal protocol databases.
* `llm_output`: Response generated by the fine-tuned LLM.
* `requires_human_approval`: Flag set when safety checks detect high-risk suggestions.
* `audit_payload`: Log metadata tracking full execution lifecycle.

**Nodes**

* **`parse_and_validate_input`:** Extracts `patient_id` from incoming metadata or parses it from `user_query` using named entity recognition/regex.
* **`request_patient_id`:** Formats a response prompting the user to specify a patient when no `patient_id` is found.
* **`fetch_ehr_context`:** Queries EHR databases using `patient_id` to retrieve history and check pending tests.
* **`retrieve_protocols`:** Vector search node fetching internal medical guidelines to guarantee explainability.
* **`generate_llm_response`:** Synthesizes context, protocols, and query using the fine-tuned LLM.
* **`guardrail_evaluator`:** Inspects output to block direct unvalidated prescriptions or unauthorized procedures.
* **`human_validation_gate`:** Holds execution for physician sign-off on flagged high-risk actions.
* **`audit_logger`:** Writes execution traces and validation states to audit storage.

**Graph Edges & Execution Flow**

* **`START`** $\rightarrow$ **`parse_and_validate_input`**
* **Conditional Edge (`parse_and_validate_input`):**
* If `missing_patient_id == True` $\rightarrow$ **`request_patient_id`** $\rightarrow$ **`END`**
* If `missing_patient_id == False` $\rightarrow$ **`fetch_ehr_context`** $\rightarrow$ **`retrieve_protocols`** $\rightarrow$ **`generate_llm_response`** $\rightarrow$ **`guardrail_evaluator`**


* **Conditional Edge (`guardrail_evaluator`):**
* If `requires_human_approval == True` $\rightarrow$ **`human_validation_gate`** $\rightarrow$ **`audit_logger`** $\rightarrow$ **`END`**
* If `requires_human_approval == False` $\rightarrow$ **`audit_logger`** $\rightarrow$ **`END`**

In [ ]:
from src.model_loading import model, tokenizer

In [ ]:
from src.graph_state import GraphState

## Mock Patient Database

This `mock_patient_db` is a list of dictionaries, where each dictionary represents a patient record. Each patient record includes:

*   `patient_id`: A unique identifier for the patient.
*   `medical_history`: A list of medical conditions the patient has.
*   `current_medications`: A list of medications the patient is currently taking.
*   `pending_tests`: A list of any tests that are pending for the patient.

In [ ]:
from src.patient_database import mock_patient_db

### Node: `parse_and_validate_input`

This node is responsible for initializing the workflow by attempting to identify the `patient_id` from the user's input.

**Functionality:**
1.  **Extracts `user_query`**: It retrieves the raw prompt provided by the physician.
2.  **Identifies `patient_id`**:
    *   If a `patient_id` is already present in the `state` (e.g., from session metadata), it uses that.
    *   Otherwise, it tries to extract a `patient_id` from the `user_query` using a regular expression (e.g., finding patterns like `P001`, `P123`).
3.  **Sets `missing_patient_id` flag**: A boolean flag is set to `True` if no `patient_id` could be found or extracted, indicating that the system cannot proceed without this critical piece of information.
4.  **Updates `audit_payload`**: Logs whether the input was parsed and if a patient ID was found.

In [ ]:
from src.input_nodes import parse_and_validate_input

### Node: `request_patient_id`

This node is activated when the `parse_and_validate_input` node determines that no `patient_id` could be found or extracted from the user's query or session metadata.

**Functionality:**
1.  **Sets `llm_output`**: It generates a user-friendly message prompting the medical professional to provide the patient's identifier.
2.  **Sets `missing_patient_id`**: It maintains the `missing_patient_id` flag as `True` to reflect that the patient context is still unresolved.
3.  **Updates `audit_payload`**: Logs that a request for a patient ID has been issued.

In [ ]:
from src.input_nodes import request_patient_id

### Node: `fetch_ehr_context`

This node simulates querying a structured Electronic Health Record (EHR) database to retrieve relevant patient information based on the `patient_id`.

**Functionality:**
1.  **Retrieves `patient_id`**: It takes the `patient_id` from the current graph state.
2.  **Searches Mock Database**: It iterates through the `mock_patient_db` to find a matching patient record.
3.  **Extracts Context**: If a patient is found, it extracts their `medical_history`, `current_medications`, and `pending_tests`.
4.  **Handles Missing Patient**: If the `patient_id` is not found in the mock database, it prints a warning.
5.  **Updates `patient_context`**: The retrieved patient information is stored in the `patient_context` field of the graph state.

In [ ]:
from src.ehr_nodes import fetch_ehr_context

### Mock Clinical Protocols and ChromaDB Vector Store

This section defines a set of `mock_protocols` representing various clinical guidelines. These protocols are then embedded using the pre-loaded language model and stored in an in-memory ChromaDB vector store (`protocols_collection`) for efficient semantic search by the `retrieve_protocols` node.

In [ ]:
from src.protocols_database import mock_protocols

In [ ]:
from src.vectorstore import retriever

### Node: `retrieve_protocols`

This node is responsible for fetching relevant clinical protocols from the ChromaDB vector store based on the `user_query`.

**Functionality:**
1.  **Embeds Query**: It generates an embedding for the `user_query` using the same `get_embedding` function used for the protocols.
2.  **Queries ChromaDB**: It performs a similarity search against the `protocols_collection` in ChromaDB.
3.  **Retrieves Documents**: It extracts the `content` of the top-N most relevant protocols.
4.  **Updates `retrieved_docs`**: The content of the retrieved protocols is added to the `retrieved_docs` list in the graph state, making them available for the LLM.

In [ ]:
from src.retrieval_nodes import retrieve_protocols

### Node: `generate_llm_response`

This node is the core generation component of the pipeline. It synthesizes all available information to formulate a response to the user's query.

**Functionality:**
1.  **Gathers Context**: It collects the `user_query`, `patient_context` (EHR data), and `retrieved_docs` (clinical protocols) from the graph state.
2.  **Constructs Prompt**: It creates a comprehensive prompt for the LLM, integrating the system instruction, user query, patient's EHR, and relevant protocols. The prompt follows the Alpaca template used during the model's fine-tuning.
3.  **Generates Response**: It uses the `meditron` model (wrapped via LangChain's `HuggingFacePipeline`) to generate a response based on the constructed prompt.
4.  **Updates `llm_output`**: The generated response is stored in the `llm_output` field of the graph state.

In [ ]:
from src.generation_nodes import generate_llm_response

### Node: `guardrail_evaluator`

This node acts as a safety guardrail, inspecting the LLM's generated response for sensitive keywords or phrases that might indicate a need for human oversight before delivery to the user.

**Functionality:**
1.  **Retrieves `llm_output`**: It takes the generated response from the `generate_llm_response` node.
2.  **Identifies Sensitive Keywords**: It checks the `llm_output` against a predefined list of sensitive medical terms (e.g., "prescription," "diagnose," "surgery").
3.  **Sets `requires_human_approval` flag**: If any sensitive keyword is detected, this flag is set to `True`, indicating that the LLM's output requires review by a medical professional before being presented.
4.  **Updates `audit_payload`**: Logs whether guardrail evaluation was performed and the resulting `requires_human_approval` status.

In [ ]:
from src.guardrail_nodes import guardrail_evaluator

### Node: `human_validation_gate`

This node represents a critical human-in-the-loop step in the workflow, triggered when the `guardrail_evaluator` flags an LLM response as requiring human approval.

**Functionality (Simulated):**
1.  **Detects Approval Requirement**: It activates when `requires_human_approval` is `True`.
2.  **Simulates Pause**: In a real-world scenario, this node would pause the automated workflow and trigger an alert to a human medical professional for review and approval.
3.  **Informs User**: For this simulation, it modifies the `llm_output` to a message indicating that human review is pending, preventing potentially sensitive or unauthorized information from being directly delivered.
4.  **Preserves Original Output**: It stores the original `llm_output` within the `audit_payload` for the human reviewer to examine.
5.  **Updates `audit_payload`**: Logs that the human validation gate was triggered and includes the original LLM output for review.

In [ ]:
from src.guardrail_nodes import human_validation_gate

In [ ]:
from src.audit_nodes import audit_logger

In [ ]:
from IPython.display import Image, display

from src.graph import app

# Draw the graph
print("\n--- Visualizing the LangGraph ---")
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
import src.examples